## Ładowanie danych

In [ ]:
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
import pandas as pd
from pandas import DataFrame
from pandas import Series

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

columns_to_load: list[str] = [
    "LB", "AC.1", "FM.1", "UC.1", "DL.1", "DS.1", "DP.1", "ASTV",
    "MSTV", "ALTV", "MLTV", "Width", "Min", "Max", "Nmax", "Nzeros", 
    "Mode", "Mean", "Median", "Variance", "Tendency", "CLASS"
]

df: pd.DataFrame = pd.read_excel(
    "CTG.xls", sheet_name="Data", skiprows=1, usecols=columns_to_load)

df.rename(columns={col: col.replace('.1', '')
          for col in df.columns if '.1' in col}, inplace=True)

## Eksploracja danych

In [ ]:
stats: DataFrame = df.describe().T
missing = df.isnull().sum()
stats['missing'] = missing
stats

,count,mean,std,min,25%,50%,75%,max,missing
LB,2126.0,133.303857,9.840844,106.0,126.000000,133.000000,140.000000,160.000000,3
AC,2126.0,0.003170,0.003860,0.0,0.000000,0.001630,0.005631,0.019284,3
FM,2127.0,0.009695,0.047764,0.0,0.000000,0.000000,0.002527,0.480634,2
UC,2127.0,0.004362,0.002949,0.0,0.001877,0.004484,0.006527,0.014925,2
DL,2128.0,0.001890,0.002976,0.0,0.000000,0.000000,0.003270,0.015385,1
DS,2128.0,0.000004,0.000069,0.0,0.000000,0.000000,0.000000,0.001353,1
DP,2128.0,0.000159,0.000590,0.0,0.000000,0.000000,0.000000,0.005348,1
ASTV,2127.0,47.008933,17.210648,12.0,32.000000,49.000000,61.000000,87.000000,2
MSTV,2127.0,1.335449,0.891543,0.2,0.700000,1.200000,1.700000,7.000000,2
ALTV,2127.0,9.884814,18.476534,0.0,0.000000,0.000000,11.000000,91.000000,2


- Zbiór danych zawiera 2126 obserwacji i 21 cech diagnostycznych oraz kolumnę z klasą (CLASS).
- Brakujące dane: W większości cech występują pojedyncze brakujące wartości (1–3 na cechę), co stanowi bardzo mały odsetek i można je łatwo uzupełnić lub usunąć.
- Statystyki cech: Większość cech ma wartości bliskie zeru lub niskie średnie, co sugeruje, że dane są zróżnicowane i mogą wymagać standaryzacji.
- Kolumna CLASS: Obejmuje 10 klas (od 1 do 10), średnia to 4,5, a rozkład klas należy sprawdzić pod kątem zbalansowania.

Wnioski: Dane są kompletne, dobrze opisane i gotowe do dalszej analizy po uzupełnieniu braków. Warto rozważyć normalizację cech przed budową modeli klasyfikacyjnych.

In [ ]:
class_counts: Series[int] = df['CLASS'].value_counts().sort_index()
print("Liczność poszczególnych klas:\n", class_counts)

Liczność poszczególnych klas:
 CLASS
1.0     384
2.0     579
3.0      53
4.0      81
5.0      72
6.0     332
7.0     252
8.0     107
9.0      69
10.0    197
Name: count, dtype: int64


Liczności poszczególnych klas nie są zbalansowane. Klasy 2.0, 1.0 i 6.0 mają najwyższą liczność, a 3.0, 5.0, 9.0 najniższą. Może to utrudnić skuteczną klasyfikację metod przez niezbalansowane dane i warto będzie skorzystać z metody korygującej to.